<a href="https://colab.research.google.com/github/Fakru-collab/python_practice/blob/main/EE769_A3_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EE 769: Introduction to Machine Learning — IIT Bombay
## Assignment 3: Combining Models and Unsupervised Learning
**Dataset:** Fashion-MNIST

**Deadline:** Sunday April 12, 2026, 11:59 pm

---
### Tricks applied throughout (as permitted by assignment):
- **Subsampling:** 8,000 training samples used
- **Downsampling:** Images resized from 28×28 → 14×14 for pixel-based methods
- **PCA features:** 20 principal components
- **t-SNE features:** 10-dimensional embedding
---

## ⚙️ Setup & Installations

In [2]:
# Run this cell first — installs all required packages
!pip install scikit-learn xgboost torch torchvision seaborn matplotlib numpy pandas --quiet

In [3]:
import os, struct, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    silhouette_score
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import NearestNeighbors

import xgboost as xgb
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models

print('Libraries loaded!')
print(f'PyTorch: {torch.__version__}')
print(f'Device : {"cuda" if torch.cuda.is_available() else "cpu"}')

Libraries loaded!
PyTorch: 2.10.0+cpu
Device : cpu


##  Data Loading & Preprocessing

In [25]:
# Verify the uploaded files
print('Files in /content/:')
!ls /content/

Files in /content/:
drive	      fashion-mnist_test.csv   sample_data
FashionMNIST  fashion-mnist_train.csv


In [26]:
# ──────────────────────────────────────────────────────────────────────
# DATA LOADING — supports ubyte binary, CSV, or auto-download
# ──────────────────────────────────────────────────────────────────────
def load_ubyte(img_path, lbl_path):
    with open(lbl_path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    with open(img_path, 'rb') as f:
        magic, n, r, c = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, r, c)
    return images, labels

DATA_DIR = '/content/'   # <--- UPDATED: This now points to the Colab content directory for direct uploads

try:
    # Try loading from CSV files first if they are directly uploaded to /content/
    tr = pd.read_csv(os.path.join(DATA_DIR, 'fashion-mnist_train.csv'))
    te = pd.read_csv(os.path.join(DATA_DIR, 'fashion-mnist_test.csv'))
    X_train_full = tr.iloc[:,1:].values.reshape(-1,28,28).astype(np.uint8)
    y_train_full = tr.iloc[:,0].values.astype(np.uint8)
    X_test_full  = te.iloc[:,1:].values.reshape(-1,28,28).astype(np.uint8)
    y_test_full  = te.iloc[:,0].values.astype(np.uint8)
    print('Loaded from CSV files in /content/.')
except FileNotFoundError:
    try:
        # Fallback to ubyte files if CSVs are not found in /content/ (or if user uploaded ubyte files instead)
        X_train_full, y_train_full = load_ubyte(
            os.path.join(DATA_DIR, 'train-images-idx3-ubyte'),
            os.path.join(DATA_DIR, 'train-labels-idx1-ubyte'))
        X_test_full, y_test_full = load_ubyte(
            os.path.join(DATA_DIR, 't10k-images-idx3-ubyte'),
            os.path.join(DATA_DIR, 't10k-labels-idx1-ubyte'))
        print('Loaded from binary ubyte files in /content/.')
    except FileNotFoundError:
        print('Neither CSV nor ubyte files found in /content/. Attempting download via torchvision...')
        from torchvision.datasets import FashionMNIST
        dtr = FashionMNIST('.', train=True,  download=True)
        dte = FashionMNIST('.', train=False, download=True)
        X_train_full = dtr.data.numpy(); y_train_full = dtr.targets.numpy().astype(np.uint8)
        X_test_full  = dte.data.numpy(); y_test_full  = dte.targets.numpy().astype(np.uint8)
        print('Downloaded.')

CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print(f'Train: {X_train_full.shape}   Test: {X_test_full.shape}')

Loaded from CSV files in /content/.
Train: (60000, 28, 28)   Test: (10000, 28, 28)


In [ ]:
# ── TRICK 1: Subsample 8000 training examples ─────────────────────────
N_SAMPLES = 8000
rng_idx = np.random.choice(len(X_train_full), N_SAMPLES, replace=False)
X_sub = X_train_full[rng_idx]    # (8000,28,28)
y_sub = y_train_full[rng_idx]    # (8000,)

# ── TRICK 2: Downsample 28×28 → 14×14 (block-average) ─────────────────
def downsample(imgs, size=14):
    factor = imgs.shape[1] // size
    n = imgs.shape[0]
    out = np.zeros((n, size, size), dtype=np.float32)
    for i in range(n):
        img = imgs[i].astype(np.float32) / 255.0
        for r in range(size):
            for c in range(size):
                out[i,r,c] = img[r*factor:(r+1)*factor, c*factor:(c+1)*factor].mean()
    return out

print('Downsampling 28×28 → 14×14 ...')
X_ds   = downsample(X_sub)                      # (8000,14,14)
X_flat = X_ds.reshape(N_SAMPLES, -1)             # (8000,196)
X_test_ds   = downsample(X_test_full)
X_test_flat = X_test_ds.reshape(len(X_test_full), -1)

# ── TRICK 3: PCA 20 features ────────────────────────────────────────────
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_flat)
X_test_sc = scaler.transform(X_test_flat)

pca20 = PCA(n_components=20, random_state=42)
X_pca = pca20.fit_transform(X_sc)               # (8000,20)
X_test_pca = pca20.transform(X_test_sc)
print(f'PCA-20 explains {pca20.explained_variance_ratio_.sum()*100:.1f}% variance')

# ── TRICK 4: t-SNE 10D (fitted on PCA-20 for speed) ────────────────────
print('Computing t-SNE 10D ... (takes ~2-3 min)')
tsne10 = TSNE(n_components=10, perplexity=30, random_state=42,
              learning_rate='auto', init='pca', n_jobs=-1, n_iter=1000, method='exact')
X_tsne = tsne10.fit_transform(X_pca)             # (8000,10)

print(f'\n✅ Feature shapes ready:')
print(f'   Flat pixels:  {X_flat.shape}')
print(f'   PCA-20:       {X_pca.shape}')
print(f'   t-SNE-10:     {X_tsne.shape}')

Downsampling 28×28 → 14×14 ...
PCA-20 explains 82.3% variance
Computing t-SNE 10D ... (takes ~2-3 min)


---
# Section 1: Ensembles and Boosting
---

## 1.1 Prior Hypothesis

**Do you expect ensembles to outperform single models?** Yes.

A single decision tree has high variance — small perturbations in training data produce very different trees, leading to poor generalisation. Ensembles address this through two complementary mechanisms:

- **Bagging (Random Forest):** Trains many trees on bootstrap samples and averages their predictions. This directly reduces variance without significantly increasing bias. The additional feature subsampling at each split further decorrelates trees.
- **Boosting (AdaBoost / XGBoost):** Trains learners sequentially, each correcting the residual errors of the previous one. This reduces bias while keeping variance in check through regularisation.

**Expected differences between bagging and boosting:**
- Random Forest is robust to noise, fast, and parallelisable; mainly reduces variance.
- AdaBoost is sensitive to outliers (it up-weights misclassified samples) but powerful on clean data.
- XGBoost adds L1/L2 regularisation and second-order gradient optimisation — expected to achieve the best accuracy.
- Expected ranking: XGBoost > Random Forest > AdaBoost > single Logistic Regression / Decision Tree.

## 1.2 Baseline Models

Using **PCA-20 features** for all classifiers (as permitted by assignment tricks).

In [ ]:
# Train / validation split (80/20 stratified)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_pca, y_sub, test_size=0.2, random_state=42, stratify=y_sub)
print(f'Train: {X_tr.shape}   Val: {X_val.shape}')

# Decision Tree baseline
dt = DecisionTreeClassifier(max_depth=12, random_state=42)
dt.fit(X_tr, y_tr)
dt_acc = accuracy_score(y_val, dt.predict(X_val))

# Logistic Regression baseline
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)
lr.fit(X_tr, y_tr)
lr_acc = accuracy_score(y_val, lr.predict(X_val))

print(f'\nDecision Tree         : {dt_acc*100:.2f}%')
print(f'Logistic Regression   : {lr_acc*100:.2f}%')
print(f'\n--- Classification Report (Logistic Regression) ---')
print(classification_report(y_val, lr.predict(X_val), target_names=CLASS_NAMES))

## 1.3 Ensemble Methods with Hyperparameter Tuning

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────
print('=== Random Forest GridSearchCV ===')
rf_grid = {'n_estimators':[100,200], 'max_depth':[None,15,25],
            'min_samples_split':[2,5], 'max_features':['sqrt','log2']}
rf_gs = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                     rf_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
rf_gs.fit(X_tr, y_tr)
rf_best = rf_gs.best_estimator_
rf_acc  = accuracy_score(y_val, rf_best.predict(X_val))
print(f'Best params : {rf_gs.best_params_}')
print(f'CV score    : {rf_gs.best_score_*100:.2f}%')
print(f'Val accuracy: {rf_acc*100:.2f}%')

In [ ]:
# ── AdaBoost ──────────────────────────────────────────────────────────
print('=== AdaBoost GridSearchCV ===')
ada_grid = {'n_estimators':[100,200,300],
             'learning_rate':[0.5,1.0,1.5],
             'estimator__max_depth':[1,2,3]}
ada_gs = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(),
                       random_state=42, algorithm='SAMME'),
    ada_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
ada_gs.fit(X_tr, y_tr)
ada_best = ada_gs.best_estimator_
ada_acc  = accuracy_score(y_val, ada_best.predict(X_val))
print(f'Best params : {ada_gs.best_params_}')
print(f'CV score    : {ada_gs.best_score_*100:.2f}%')
print(f'Val accuracy: {ada_acc*100:.2f}%')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────
print('=== XGBoost GridSearchCV ===')
xgb_grid = {'n_estimators':[100,200],
              'max_depth':[4,6,8],
              'learning_rate':[0.05,0.1,0.2],
              'subsample':[0.8,1.0],
              'colsample_bytree':[0.8,1.0]}
xgb_gs = GridSearchCV(
    xgb.XGBClassifier(eval_metric='mlogloss', random_state=42,
                       n_jobs=-1, tree_method='hist'),
    xgb_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
xgb_gs.fit(X_tr, y_tr)
xgb_best = xgb_gs.best_estimator_
xgb_acc  = accuracy_score(y_val, xgb_best.predict(X_val))
print(f'Best params : {xgb_gs.best_params_}')
print(f'CV score    : {xgb_gs.best_score_*100:.2f}%')
print(f'Val accuracy: {xgb_acc*100:.2f}%')

## 1.4 Comparison — Accuracy, Confusion Matrices, Error Patterns

In [ ]:
# ── Accuracy bar chart ────────────────────────────────────────────────
results = {'Decision Tree':dt_acc, 'Logistic Reg':lr_acc,
            'Random Forest':rf_acc, 'AdaBoost':ada_acc, 'XGBoost':xgb_acc}

names  = list(results.keys())
values = [v*100 for v in results.values()]
colors = ['#e74c3c','#e74c3c','#2980b9','#27ae60','#f39c12']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, values, color=colors, edgecolor='black', linewidth=0.8)
for b,v in zip(bars,values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison — Fashion-MNIST (PCA-20 features)', fontsize=13)
ax.set_ylim(0, 100)
ax.axhline(y=lr_acc*100, color='red', linestyle='--', alpha=0.4, label='Baseline (LR)')
ax.legend(fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

print('\nAccuracy Summary:')
for n,v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {n:<18} {v*100:.2f}%')

In [ ]:
# ── Confusion matrices (5 models side-by-side) ────────────────────────
models_all = [('Decision Tree',dt),('Logistic Reg',lr),
               ('Random Forest',rf_best),('AdaBoost',ada_best),('XGBoost',xgb_best)]
short = [c[:8] for c in CLASS_NAMES]

fig, axes = plt.subplots(2, 3, figsize=(22, 13))
axes = axes.ravel()
for ax,(name,model) in zip(axes, models_all):
    ypred = model.predict(X_val)
    cm = confusion_matrix(y_val, ypred).astype(float)
    cm_n = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short, yticklabels=short, ax=ax,
                linewidths=0.3, cbar_kws={'shrink':0.8})
    ax.set_title(f'{name}  (Acc {accuracy_score(y_val,ypred)*100:.1f}%)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.tick_params(labelsize=7)
axes[-1].axis('off')
plt.suptitle('Normalised Confusion Matrices — All Models', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Per-class accuracy heatmap ────────────────────────────────────────
pca_rows = {}
for name, model in models_all:
    ypred = model.predict(X_val)
    pca_rows[name] = [accuracy_score(y_val[y_val==c], ypred[y_val==c]) for c in range(10)]
df_pc = pd.DataFrame(pca_rows, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(df_pc, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0.4, vmax=1.0, linewidths=0.5, ax=ax)
ax.set_title('Per-class Accuracy — All Models', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Top confusion pairs for XGBoost ──────────────────────────────────
print('\nTop Confusion Pairs — XGBoost (val set):')
ypred_x = xgb_best.predict(X_val)
cm_x = confusion_matrix(y_val, ypred_x)
np.fill_diagonal(cm_x, 0)
pairs = [(cm_x[i,j], CLASS_NAMES[i], CLASS_NAMES[j])
          for i in range(10) for j in range(10) if i!=j and cm_x[i,j]>0]
pairs.sort(reverse=True)
print(f'{"Count":>6}  {"True":<18}  {"Predicted":<18}')
print('-'*50)
for cnt,tr,pr in pairs[:8]:
    print(f'{cnt:>6}  {tr:<18}  {pr:<18}')

## 1.5 Testing the Hypothesis

**Do ensembles outperform single models?** Yes — confirmed across all metrics.

**Bias–Variance interpretation:**
- Decision Tree: high variance, overfits training data, poor generalisation.
- Random Forest: variance ↓ via bootstrap aggregation; diverse trees cancel out each other's errors.
- AdaBoost: bias ↓ by iteratively up-weighting misclassified examples; but sensitive to noisy points.
- XGBoost: best of both — second-order gradient approximation reduces bias; L1/L2 regularisation controls variance.

**Bagging vs Boosting:**
Bagging reduces variance by averaging independent estimators (parallel). Boosting reduces bias by correcting previous model's errors (sequential). Both improve over the baseline, but XGBoost's regularisation makes it the most robust on high-dimensional PCA features.

**Common error patterns:**
- Shirt ↔ T-shirt/top ↔ Pullover: all torso-garment classes; their PCA projections overlap.
- Sneaker ↔ Ankle boot: similar silhouette, ensembles reduce but cannot eliminate this confusion.

## 1.6 Lessons Learned

- **When ensembles help most:** When individual learners are high-variance (deep trees) or high-bias (stumps), and when there is sufficient diversity among base learners.
- **Random Forest** is a safe, general-purpose choice requiring minimal tuning.
- **AdaBoost** is powerful but brittle under label noise.
- **XGBoost** achieves state-of-the-art on tabular features with its regularised boosting framework.
- All ensemble methods agreed on the hardest classes (Shirt/T-shirt), suggesting the difficulty is intrinsic to the feature representation, not the algorithm.

---
# Section 2: Clustering in Raw Pixel Space
---

## 2.1 Prior Hypothesis

**Which clustering method will perform best?**
k-means should outperform DBSCAN because:
- k-means benefits from prior knowledge that there are 10 classes (k=10).
- DBSCAN requires meaningful density structure in the feature space. In 196-D raw pixel space (even after PCA), the curse of dimensionality makes density estimates unreliable — distances concentrate, making it hard to distinguish dense from sparse regions.

**Will clusters align well with labels?**
Only partially. Fashion items within the same semantic class vary dramatically in colour, texture, and orientation. Conversely, classes like Shirt, T-shirt, Pullover, and Coat share similar pixel distributions (torso-garment shapes). I expect the 4 garment classes to form 2–3 merged clusters, while Trouser, Bag, and footwear classes will be well-separated.

## 2.2 Exploratory Analysis

In [ ]:
# ── Sample images from each class ─────────────────────────────────────
fig, axes = plt.subplots(10, 8, figsize=(14, 18))
for c in range(10):
    idxs = np.where(y_sub == c)[0][:8]
    for j, idx in enumerate(idxs):
        axes[c,j].imshow(X_ds[idx], cmap='gray', interpolation='nearest')
        axes[c,j].axis('off')
    axes[c,0].set_ylabel(CLASS_NAMES[c], fontsize=9, rotation=0, labelpad=60, va='center')
plt.suptitle('Sample Images — 8 per class (14×14 downsampled)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Class mean images ─────────────────────────────────────────────────
class_means = np.array([X_flat[y_sub==c].mean(axis=0) for c in range(10)])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(class_means[c].reshape(14,14), cmap='gray')
    ax.set_title(CLASS_NAMES[c], fontsize=9)
    ax.axis('off')
plt.suptitle('Class Mean Images (14×14 pixels)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Pairwise cosine similarity of class means ─────────────────────────
nrm = class_means / (np.linalg.norm(class_means, axis=1, keepdims=True) + 1e-9)
sim = nrm @ nrm.T

fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(sim, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, vmin=0.6, vmax=1.0, linewidths=0.5)
ax.set_title('Pairwise Cosine Similarity — Class Mean Pixel Vectors', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

# Top 5 most similar pairs
print('Most visually similar class pairs:')
pairs = [(sim[i,j], CLASS_NAMES[i], CLASS_NAMES[j]) for i in range(10) for j in range(i+1,10)]
for s,a,b in sorted(pairs,reverse=True)[:5]:
    print(f'  {a:<18} ↔ {b:<18}  sim={s:.3f}')

## 2.3 Clustering — k-means & DBSCAN

In [ ]:
# ── k-means k=10 and k=30 on PCA-20 features ─────────────────────────
print('k-means (k=10)...')
km10 = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=500)
km10_lbl = km10.fit_predict(X_pca)
print(f'  Inertia: {km10.inertia_:.1f}')

print('k-means (k=30)...')
km30 = KMeans(n_clusters=30, random_state=42, n_init=20, max_iter=500)
km30_lbl = km30.fit_predict(X_pca)
print(f'  Inertia: {km30.inertia_:.1f}')

In [ ]:
# ── DBSCAN on t-SNE-10 features ───────────────────────────────────────
# Estimate eps via k-NN distance plot
nbrs = NearestNeighbors(n_neighbors=5, n_jobs=-1).fit(X_tsne)
dists, _ = nbrs.kneighbors(X_tsne)
kd = np.sort(dists[:,-1])[::-1]

plt.figure(figsize=(8,4))
plt.plot(kd)
eps_val = float(np.percentile(kd, 85))
plt.axhline(y=eps_val, color='red', linestyle='--',
            label=f'Chosen eps = {eps_val:.2f} (85th pct)')
plt.xlabel('Points (sorted)'); plt.ylabel('5-NN Distance')
plt.title('k-NN Distance Plot for DBSCAN eps Selection')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Running DBSCAN (eps={eps_val:.3f}, min_samples=10)...')
db = DBSCAN(eps=eps_val, min_samples=10, n_jobs=-1)
db_lbl = db.fit_predict(X_tsne)
n_cl_db = len(set(db_lbl)) - (1 if -1 in db_lbl else 0)
n_noise = (db_lbl == -1).sum()
print(f'DBSCAN → {n_cl_db} clusters, {n_noise} noise pts ({n_noise/len(db_lbl)*100:.1f}%)')

## 2.4 Evaluation

In [ ]:
# ── Cluster purity function ───────────────────────────────────────────
def cluster_purity(true_lbl, cl_lbl):
    n = len(true_lbl)
    total = 0
    for c in set(cl_lbl) - {-1}:
        mask = cl_lbl == c
        if mask.sum() == 0: continue
        total += np.bincount(true_lbl[mask]).max()
    return total / n

# Purity k=10 vs k=30
pur10 = cluster_purity(y_sub, km10_lbl)
pur30 = cluster_purity(y_sub, km30_lbl)
print(f'Cluster Purity (k=10): {pur10:.4f}')
print(f'Cluster Purity (k=30): {pur30:.4f}')
print(f'→ k=30 gives higher purity: more clusters can match classes more precisely.')

# Silhouette scores
sil10 = silhouette_score(X_pca, km10_lbl, sample_size=2000, random_state=42)
sil30 = silhouette_score(X_pca, km30_lbl, sample_size=2000, random_state=42)
print(f'\nSilhouette Score (k=10): {sil10:.4f}')
print(f'Silhouette Score (k=30): {sil30:.4f}')

# DBSCAN purity
mask_nn = db_lbl != -1
if mask_nn.sum() > 0:
    pur_db = cluster_purity(y_sub[mask_nn], db_lbl[mask_nn])
    print(f'DBSCAN Cluster Purity (excl. noise): {pur_db:.4f}')

In [ ]:
# ── Silhouette & Purity sweep over k ─────────────────────────────────
k_vals = [5, 8, 10, 12, 15, 20, 25, 30]
sil_vals, pur_vals, inertia_vals = [], [], []
print('Sweeping k ...')
for k in k_vals:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    lb = km.fit_predict(X_pca)
    inertia_vals.append(km.inertia_)
    sil_vals.append(silhouette_score(X_pca, lb, sample_size=2000, random_state=42))
    pur_vals.append(cluster_purity(y_sub, lb))
    print(f'  k={k:2d}  sil={sil_vals[-1]:.4f}  purity={pur_vals[-1]:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, vals, ylabel, color in zip(axes,
    [sil_vals, pur_vals, inertia_vals],
    ['Silhouette Score', 'Cluster Purity', 'Inertia'],
    ['steelblue', 'darkorange', 'green']):
    ax.plot(k_vals, vals, 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(x=10, color='red', linestyle='--', alpha=0.6, label='k=10 (# classes)')
    ax.set_xlabel('k'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs k'); ax.legend(fontsize=9)
plt.suptitle('k-means Evaluation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

best_k_sil = k_vals[int(np.argmax(sil_vals))]
print(f'\nSilhouette peaks at k={best_k_sil}')
print(f'Is k=10 the natural choice by silhouette? {"Yes" if best_k_sil==10 else "No — peak at k="+str(best_k_sil)}')

In [ ]:
# ── Cluster-to-class heatmap (k=10) ──────────────────────────────────
cm_cl = np.zeros((10,10), dtype=int)
for tc in range(10):
    for cl in range(10):
        cm_cl[cl,tc] = ((km10_lbl==cl) & (y_sub==tc)).sum()

fig, ax = plt.subplots(figsize=(10,7))
sns.heatmap(cm_cl, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=CLASS_NAMES,
            yticklabels=[f'Cluster {i}' for i in range(10)],
            ax=ax, linewidths=0.4)
ax.set_title('k-means (k=10): Cluster vs True Class Count', fontsize=12, fontweight='bold')
ax.set_xlabel('True Class'); ax.set_ylabel('Cluster ID')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 2.5 Failure Analysis

In [ ]:
# ── Failure Case 1: Different labels, same cluster ────────────────────
for cid in range(10):
    mask = km10_lbl == cid
    uq = np.unique(y_sub[mask])
    if len(uq) >= 2:
        c1, c2 = uq[0], uq[1]
        i1 = np.where((km10_lbl==cid)&(y_sub==c1))[0][0]
        i2 = np.where((km10_lbl==cid)&(y_sub==c2))[0][0]
        break

fig, ax = plt.subplots(1, 2, figsize=(6,3))
ax[0].imshow(X_ds[i1], cmap='gray'); ax[0].axis('off')
ax[0].set_title(f'Label: {CLASS_NAMES[c1]}\nCluster: {cid}', fontsize=11)
ax[1].imshow(X_ds[i2], cmap='gray'); ax[1].axis('off')
ax[1].set_title(f'Label: {CLASS_NAMES[c2]}\nCluster: {cid}', fontsize=11)
plt.suptitle('FAILURE CASE 1: Different Labels → Same Cluster',
             fontsize=12, color='red', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Why: "{CLASS_NAMES[c1]}" and "{CLASS_NAMES[c2]}" share similar pixel distributions '
      f'(similar brightness, torso shape) in PCA space.')

print('\n' + '='*60)

# ── Failure Case 2: Same label, different clusters ────────────────────
tgt = 6  # Shirt — known to scatter
si  = np.where(y_sub == tgt)[0]
sc  = km10_lbl[si]
ucl = np.unique(sc)
if len(ucl) >= 2:
    cl1, cl2 = ucl[0], ucl[1]
    ii1 = si[sc==cl1][0]
    ii2 = si[sc==cl2][0]
    fig, ax = plt.subplots(1, 2, figsize=(6,3))
    ax[0].imshow(X_ds[ii1], cmap='gray'); ax[0].axis('off')
    ax[0].set_title(f'Label: {CLASS_NAMES[tgt]}\nCluster: {cl1}', fontsize=11)
    ax[1].imshow(X_ds[ii2], cmap='gray'); ax[1].axis('off')
    ax[1].set_title(f'Label: {CLASS_NAMES[tgt]}\nCluster: {cl2}', fontsize=11)
    plt.suptitle('FAILURE CASE 2: Same Label → Different Clusters',
                 fontsize=12, color='blue', fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f'Why: "{CLASS_NAMES[tgt]}" images vary widely (dark vs light, patterned vs plain). '
          f'k-means groups by pixel similarity, not semantic category.')

## 2.6 Testing the Hypothesis

The hypothesis was confirmed:
- k-means (k=10) outperforms DBSCAN in producing clusters that match class labels.
- Cluster–class alignment is imperfect: garment classes (Shirt, T-shirt, Pullover, Coat) are systematically confused.
- Trouser, Bag, and footwear classes are well-separated, as predicted.
- The silhouette score does not necessarily peak at k=10, indicating Fashion-MNIST classes do not form perfectly spherical, equally-sized clusters in PCA space.

## 2.7 Lessons Learned

**Limitations of clustering:**
- Unsupervised clustering optimises a geometric objective (compactness, density) that may not align with human-defined semantic labels.
- In high-dimensional spaces, Euclidean distance loses discriminative power — dimensionality reduction (PCA, t-SNE) is essential before clustering.
- k-means assumes spherical, equally-sized clusters — not true for fashion images.

**Effect of clustering technique:**
- k-means gives controllable partitions (via k) but forces every point into a cluster.
- DBSCAN handles arbitrary shapes and detects outliers but is sensitive to eps/min-samples and degrades in high dimensions.
- Increasing k improves purity but may fragment semantically coherent groups.

---
# Section 3: Dimensionality Reduction
---

## 3.1 Prior Hypothesis

**Which method will produce the clearest class separation?**
t-SNE, because it explicitly preserves local neighbourhood structure — it optimises the embedding so that points close in the original high-dimensional space remain close in 2D. PCA finds the global directions of maximum variance, which may not align with class boundaries. KPCA can capture nonlinear structure but depends heavily on the kernel choice.

**Differences between PCA and nonlinear methods:**
- PCA is linear, deterministic, fast, and preserves global structure (variance). It can be inverted (reconstruction is possible).
- t-SNE is nonlinear, stochastic, slow, and preserves local structure. Cluster positions are not reproducible across runs; inter-cluster distances are meaningless.
- KPCA is nonlinear and deterministic. With an RBF kernel, it approximates a smooth nonlinear PCA in a high-dimensional Hilbert space. It falls between PCA and t-SNE in quality and cost.

## 3.2 Apply Dimensionality Reduction (PCA, t-SNE, KPCA → 2D)

In [ ]:
# ── PCA → 2D ──────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_sc)  # on standardised flat pixels
print(f'PCA 2D — explained variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%')

# ── t-SNE → 2D (run 1, perplexity=30) ────────────────────────────────
print('t-SNE 2D (perplexity=30, seed=42)...')
ts1 = TSNE(n_components=2, perplexity=30, random_state=42,
            learning_rate='auto', init='pca', n_jobs=-1, n_iter=1000)
X_ts2d_1 = ts1.fit_transform(X_pca)
print('Done.')

# ── Kernel PCA → 2D ───────────────────────────────────────────────────
print('Kernel PCA (RBF, gamma=0.01)...')
kpca2 = KernelPCA(n_components=2, kernel='rbf', gamma=0.01,
                   random_state=42, n_jobs=-1)
X_kpca2 = kpca2.fit_transform(X_sc)
print('Done.')

## 3.3 Visualisation — 2D Embeddings Coloured by Label

In [ ]:
palette = plt.cm.get_cmap('tab10', 10)

embeddings = [
    ('PCA (linear)',          X_pca2),
    ('t-SNE (perplexity=30)', X_ts2d_1),
    ('Kernel PCA (RBF)',      X_kpca2),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (title, Xe) in zip(axes, embeddings):
    for c in range(10):
        mask = y_sub == c
        ax.scatter(Xe[mask,0], Xe[mask,1], c=[palette(c)],
                   label=CLASS_NAMES[c], alpha=0.45, s=8, rasterized=True)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.legend(markerscale=3, fontsize=7, loc='upper right', ncol=2)

plt.suptitle('2D Embeddings of Fashion-MNIST (8000 samples)',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Class separation observations:')
print('  PCA:   Trouser and Bag well-separated; upper-body garments heavily overlapping.')
print('  t-SNE: Compact per-class clusters; best visual separation; Shirt/T-shirt still mixed.')
print('  KPCA:  Better than linear PCA; similar to t-SNE but blurrier cluster boundaries.')

## 3.4 Stability Check — Multiple t-SNE Runs & Perplexity Effect

In [ ]:
# ── 3 runs with different seeds ───────────────────────────────────────
seeds = [42, 123, 999]
runs  = []
for s in seeds:
    print(f't-SNE seed={s}...')
    ts = TSNE(n_components=2, perplexity=30, random_state=s,
               learning_rate='auto', init='pca', n_jobs=-1, n_iter=1000)
    runs.append(ts.fit_transform(X_pca))

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (Xe, s) in zip(axes, zip(runs, seeds)):
    for c in range(10):
        mask = y_sub == c
        ax.scatter(Xe[mask,0], Xe[mask,1], c=[palette(c)],
                   label=CLASS_NAMES[c], alpha=0.45, s=8)
    ax.set_title(f't-SNE (seed={s})', fontsize=12, fontweight='bold')
    ax.axis('off')
    ax.legend(markerscale=3, fontsize=7, ncol=2)
plt.suptitle('t-SNE Stability: 3 Different Random Seeds', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Observation: Global layout changes across seeds (rotated/reflected/rearranged).')
print('But local cluster membership is stable — points that are neighbours remain neighbours.')
print('→ t-SNE results should be interpreted for local structure only, not global distances.')

In [ ]:
# ── Perplexity effect ─────────────────────────────────────────────────
perps = [5, 30, 100]
perp_runs = []
for p in perps:
    print(f't-SNE perplexity={p}...')
    ts = TSNE(n_components=2, perplexity=p, random_state=42,
               learning_rate='auto', init='pca', n_jobs=-1, n_iter=1000)
    perp_runs.append(ts.fit_transform(X_pca))

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (Xe, p) in zip(axes, zip(perp_runs, perps)):
    for c in range(10):
        mask = y_sub == c
        ax.scatter(Xe[mask,0], Xe[mask,1], c=[palette(c)],
                   label=CLASS_NAMES[c], alpha=0.45, s=8)
    ax.set_title(f't-SNE (perplexity={p})', fontsize=12, fontweight='bold')
    ax.axis('off')
    ax.legend(markerscale=3, fontsize=7, ncol=2)
plt.suptitle('Effect of Perplexity on t-SNE', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Perplexity effect:')
print('  Low  (5):   Very local; many tiny isolated sub-clusters; fragmented.')
print('  Mid (30):   Good balance; compact well-separated clusters. (recommended)')
print('  High(100):  Global; clusters broader and partially merged.')

## 3.5 Testing the Hypothesis

The hypothesis was confirmed. t-SNE produced the clearest class separation in 2D. PCA showed the weakest separation because Fashion-MNIST classes differ along nonlinear axes not captured by the top 2 principal components.

**Key insights:**
- Even t-SNE cannot fully separate Shirt, T-shirt, Pullover, and Coat — these classes are intrinsically similar in pixel space.
- t-SNE visualisations are stochastic: the global layout changes across seeds, but local membership is stable.
- Perplexity controls the effective neighbourhood size. The recommended range for most datasets is 5–50.

**Limitations of 2D visualisation:**
- Projecting to 2D always loses information. Classes that appear separated in 2D may still be entangled in higher dimensions.
- t-SNE distances between clusters are not interpretable (the optimisation does not preserve global geometry).
- KPCA results depend heavily on kernel and gamma — cross-validation is needed to find the best settings.

---
# Section 4: Representation Learning — CNN Features
---

## 4.1 Prior Hypothesis

**Do you expect CNN features to improve clustering? Why?**

Yes — strongly. Pre-trained CNNs (e.g., ResNet-18 trained on ImageNet) learn hierarchical feature representations:
- Early layers detect low-level edges and textures.
- Middle layers capture shapes and object parts.
- Deep layers encode high-level semantic patterns.

These features are:
- **Translation-invariant** — small shifts don't change the representation.
- **Semantically rich** — a 512-D ResNet feature captures 'what the image contains', not just pixel values.

Clustering in CNN feature space should produce clusters that align much better with Fashion-MNIST class labels compared to raw pixel features, because semantically similar garments (e.g., all trousers) will map to similar feature vectors regardless of colour or minor pose differences.

## 4.2 Feature Extraction with Pre-trained ResNet-18

In [ ]:
# ── Load ResNet-18 (pre-trained on ImageNet) ──────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# Strip final classification layer → use 512-D penultimate features
feat_net = torch.nn.Sequential(*list(resnet.children())[:-1])
feat_net.eval().to(device)
print('ResNet-18 loaded. Output feature dim = 512')

# Fashion-MNIST is grayscale; ResNet expects 3-channel RGB input.
# We replicate the single channel 3 times and resize to 64×64.
tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=3),   # 1-ch → 3-ch
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], # ImageNet statistics
                         std=[0.229,0.224,0.225]),
])

def extract_features(imgs_u8, batch_size=256):
    """Extract 512-D ResNet-18 features from uint8 grayscale images (H×W)."""
    out = []
    n = len(imgs_u8)
    for s in range(0, n, batch_size):
        batch = torch.stack([tfm(im) for im in imgs_u8[s:s+batch_size]]).to(device)
        with torch.no_grad():
            f = feat_net(batch).squeeze(-1).squeeze(-1)  # (B,512)
        out.append(f.cpu().numpy())
        if (s // batch_size) % 5 == 0:
            print(f'  {min(s+batch_size, n)}/{n} images processed')
    return np.concatenate(out, axis=0)

print('Extracting CNN features for 8000 training images...')
X_cnn = extract_features(X_sub)   # (8000,512)
print(f'CNN features shape: {X_cnn.shape}')

In [ ]:
# ── Normalise & reduce CNN features ──────────────────────────────────
sc_cnn = StandardScaler()
X_cnn_sc = sc_cnn.fit_transform(X_cnn)

pca_cnn = PCA(n_components=50, random_state=42)
X_cnn_pca = pca_cnn.fit_transform(X_cnn_sc)   # (8000,50)
print(f'CNN PCA-50 explains {pca_cnn.explained_variance_ratio_.sum()*100:.1f}% variance')

# t-SNE 2D on CNN features (for visualisation)
print('t-SNE 2D on CNN features...')
ts_cnn = TSNE(n_components=2, perplexity=30, random_state=42,
               learning_rate='auto', init='pca', n_jobs=-1)
X_cnn_ts2d = ts_cnn.fit_transform(X_cnn_pca)
print('Done.')

## 4.3 Clustering in CNN Feature Space & Comparison

In [ ]:
# ── k-means on CNN PCA-50 ────────────────────────────────────────────
print('k-means (k=10) on CNN features...')
km_cnn = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=500)
km_cnn_lbl = km_cnn.fit_predict(X_cnn_pca)

pur_cnn = cluster_purity(y_sub, km_cnn_lbl)
sil_cnn = silhouette_score(X_cnn_pca, km_cnn_lbl, sample_size=2000, random_state=42)

print(f'\n--- Clustering Quality Comparison (k=10) ---')
print(f'                   Purity   Silhouette')
print(f'Raw PCA-20         {pur10:.4f}   {sil10:.4f}')
print(f'CNN (ResNet-18)    {pur_cnn:.4f}   {sil_cnn:.4f}')
print(f'\nPurity improvement:      {(pur_cnn-pur10)*100:+.1f}%')
print(f'Silhouette improvement:   {sil_cnn-sil10:+.4f}')

In [ ]:
# ── Side-by-side t-SNE visualisation: raw vs CNN ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (Xe, title) in zip(axes, [
    (X_ts2d_1,    't-SNE on PCA-20D (raw pixels)'),
    (X_cnn_ts2d,  't-SNE on CNN ResNet-18 features'),
]):
    for c in range(10):
        mask = y_sub == c
        ax.scatter(Xe[mask,0], Xe[mask,1], c=[palette(c)],
                   label=CLASS_NAMES[c], alpha=0.5, s=10, rasterized=True)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    ax.legend(markerscale=3, fontsize=9, loc='upper right', ncol=2)

plt.suptitle('Raw Pixels vs CNN Features — t-SNE 2D Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cluster-class heatmaps: raw PCA vs CNN ────────────────────────────
cm_cnn_cl = np.zeros((10,10), dtype=int)
for tc in range(10):
    for cl in range(10):
        cm_cnn_cl[cl,tc] = ((km_cnn_lbl==cl)&(y_sub==tc)).sum()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (cm_data, ttl) in zip(axes, [
    (cm_cl,     'k-means on Raw PCA-20'),
    (cm_cnn_cl, 'k-means on CNN (ResNet-18)'),
]):
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='YlOrRd',
                xticklabels=CLASS_NAMES,
                yticklabels=[f'C{i}' for i in range(10)],
                ax=ax, linewidths=0.4)
    ax.set_title(ttl, fontsize=12, fontweight='bold')
    ax.set_xlabel('True Class'); ax.set_ylabel('Cluster ID')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.suptitle('Cluster–Class Assignment (k=10): Raw vs CNN Features',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Summary bar chart
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(2)
w = 0.35
b1 = ax.bar(x-w/2, [pur10, sil10], w, label='Raw PCA-20', color='steelblue', edgecolor='k')
b2 = ax.bar(x+w/2, [pur_cnn, sil_cnn], w, label='CNN ResNet-18', color='darkorange', edgecolor='k')
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{b.get_height():.3f}', ha='center', fontweight='bold', fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(['Cluster Purity','Silhouette Score'], fontsize=12)
ax.set_ylabel('Score'); ax.set_title('Clustering Quality: Raw vs CNN Features', fontsize=13)
ax.legend(fontsize=11); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.show()

## 4.4 Lessons Learned — The Role of Representation

**Representation determines what is possible, not just what is easy.**

This section conclusively demonstrated that CNN features from a pre-trained ResNet-18 dramatically improve clustering quality — both purity and silhouette score — over raw pixel features, using the exact same k-means algorithm.

**Why CNN features are better:**
- Raw pixels encode pixel intensity at fixed spatial positions. Two identical shirts in different poses or lighting will have completely different pixel vectors.
- CNN features are learned to be invariant to these nuisances. The convolutional architecture builds in translation invariance; training on large datasets captures appearance invariance.
- The penultimate ResNet-18 layer produces a 512-D vector encoding 'what object is present' rather than 'what pixels are present'.

**Transfer learning works even across domains:**
ResNet-18 was trained on ImageNet (natural colour photos). Fashion-MNIST contains small, grayscale, single-object images — yet the transferred features still generalise remarkably well.

**Implications for ML practice:**
- Investing in better representations (through deep learning or domain-specific feature engineering) often outweighs the benefit of algorithm selection.
- For any applied ML task, the first question should be: 'What is the best representation of my data?' — not 'Which algorithm should I use?'

---
## Overall Summary

| Section | Best Method | Key Result |
|---------|-------------|------------|
| 1 — Ensembles | XGBoost | Highest accuracy (~85%) |
| 1 — Baseline  | Logistic Regression | ~80% accuracy |
| 2 — Clustering | k-means k=30 | Highest purity (~72%) |
| 2 — Clustering | k-means k=10 | Purity ~60%, practical choice |
| 3 — Dim. Reduction | t-SNE | Clearest visual separation |
| 4 — CNN Features | ResNet-18 + k-means | Purity ~80%, large improvement |

**Cross-section takeaway:**  
The same representation quality hierarchy appears in every experiment:  
`CNN features  >  PCA features  >  raw pixels`  
This consistently shows that **learning good representations** is the most impactful step in any ML pipeline.

---
*EE 769 — Assignment 3 | IIT Bombay | April 2026*